In [ ]:
!pip install -q -U transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from google.colab import files

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# 1. Configuration with specific compute dtype
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # T4 GPU ke liye behtar hai
)

print("⏳ Loading Model... Is mein 1-2 mins lag saktay hain.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

# 2. Reviewer Pipeline
reviewer = pipeline("text-generation", model=model, tokenizer=tokenizer)

def run_code_review(file_content):
    messages = [
        {"role": "system", "content": "You are a Senior Developer. Identify bugs, security risks, and provide optimized code."},
        {"role": "user", "content": f"Review this code:\n\n{file_content}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = reviewer(prompt, max_new_tokens=1024, do_sample=True, temperature=0.2)
    return outputs[0]["generated_text"].split("assistant\n")[-1]

# 3. Execution
uploaded = files.upload()
for filename in uploaded.keys():
    with open(filename, 'r') as f:
        code = f.read()
    print(f"\n🔍 Analyzing {filename}...")
    print("\n" + "="*20 + " AI INSIGHTS " + "="*20)
    print(run_code_review(code))


⏳ Loading Model... Is mein 1-2 mins lag saktay hain.


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from google.colab import files

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("⏳ Loading Model with BitsAndBytes...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)


reviewer = pipeline("text-generation", model=model, tokenizer=tokenizer)

def run_code_review(file_content):
    messages = [
        {"role": "system", "content": "You are a Senior Code Reviewer. Analyze the code for bugs, efficiency, and security."},
        {"role": "user", "content": f"Review this script, make a rep and provide insights, bugs, and how to improve it:\n\n{file_content}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = reviewer(prompt, max_new_tokens=1000, do_sample=True, temperature=0.3)
    return outputs[0]["generated_text"].split("assistant\n")[-1]

# 3. Upload & Test
# uploaded = files.upload()
# for filename in uploaded.keys():
#     with open(filename, 'r') as f:
#         code = f.read()

    # Save this as a test file for your AI Agent
test_code = """
import os
import sqlite3

# 1. Security Risk: Hardcoded Secret
API_KEY = "12345-ABCDE-67890-SECRET"

def get_user_data(user_id):
    # 2. Security Risk: SQL Injection vulnerability
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = "SELECT * FROM users WHERE id = " + user_id
    cursor.execute(query)
    return cursor.fetchone()

def calculate_area(radius):
    # 3. Logical Bug: Division by zero risk if not handled
    # 4. Efficiency: Constant value could be imported from math
    pi = 3.1415926535
    return pi * (radius ** 2)

def process_files():
    # 5. Bad Practice: Bare except block
    try:
        os.system("rm -rf /tmp/cache") # 6. Risky OS command
        with open("data.txt", "r") as f:
            print(f.read())
    except:
        pass

# 7. Bug: Calling function with wrong type
print(get_user_data(123))
"""
# print(f"\n🔍 Analyzing {filename}...")
report = run_code_review(test_code)
print("\n" + "="*30 + " REVIEW REPORT " + "="*30)
print(report)



⏳ Loading Model with BitsAndBytes...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`